# BRCA 10x slide — L2 maps of the finished ablation weights

One slide: `V1_Breast_Cancer_Block_A_Section_1_image` (10x Human Breast Cancer Block A Section 1). The other BRCA slides are not included.

These are the 2026-11-17 weights in `result_uni2_ablation_<arm>_new1066axisfix_paired_v2_seed42`, not the older A2 / A3 / A7 / A8 trees. Each panel is the L2 argmax (`pred_CT`) on the fullres H&E.

Predict first, on a GPU that is not training:

```bash
CUDA_DEVICE=2 bash code/External_val/BRCA/demo_uni2_axisfix.sh predict
```

The notebook reads one prediction file per finished arm. It does not launch a GPU job.

The no-neighbor arms (`legacy_nospatial`, `adjacent_nospatial`, `l12_balanced_nospatial`) are not shown.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (REPO / "code", REPO / "code" / "External_val" / "BRCA"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import brca_external as brca_ext

brca_ext.use_embedder("uni2")
cohort = brca_ext.COHORT

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
BRCA = REPO / "data" / "Visium" / "BRCA"
HE_MAX_SIDE = 1100
DPI = 120
MARKER_PX = 0.7

# Neighbor arms only. legacy_nospatial, adjacent_nospatial, and l12_balanced_nospatial are left out.
ARMS = [
    ("spatial_k8", "trunk: adjacent + k=8 mean"),
    ("spatial_k8_gate", "trunk + spatial gate"),
    ("l12_links_1", "trunk, L12 link weights = 1"),
    ("l12_links_2", "trunk, L12 link weights = 2"),
    ("l12_balanced", "trunk + balanced L12"),
    ("focal_gamma1", "trunk + focal γ=1, class weights kept"),
    ("balanced_sampler", "trunk + sampler, no class weight"),
]

def pred_h5ad(arm):
    return BRCA / f"pred_uni2_axisfix_{arm}" / SAMPLE / f"{SAMPLE}_stardist_pred.h5ad"

In [ ]:
he_path = cohort.sample_paths(SAMPLE)["he"]
thumb, full_w, full_h = cohort._he_thumbnail(he_path, max_side=HE_MAX_SIDE)
palette = cohort._palette()["L2"]
point_s = (MARKER_PX * 72.0 / DPI) ** 2
print(f"{SAMPLE}  HE {he_path.name}  thumb {thumb.shape}  full {full_w} x {full_h}")

loaded = []
missing = []
for arm, note in ARMS:
    path = pred_h5ad(arm)
    if not path.is_file():
        missing.append(arm)
        continue
    adata = ad.read_h5ad(path)
    loaded.append((arm, note, adata))
    print(f"{arm}: {adata.n_obs:,} nuclei  {note}")
if missing:
    raise FileNotFoundError(
        "Missing predictions for "
        + ", ".join(missing)
        + ". Run: CUDA_DEVICE=2 bash code/External_val/BRCA/demo_uni2_axisfix.sh predict"
    )


## L2 cell type on the 10x fullres image

The panels share one color scale and one legend. Each point is a StarDist nucleus colored by its L2 argmax.

In [ ]:
ncols = 3
nrows = (len(loaded) + 1 + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16.8, 4.6 * nrows), dpi=DPI)
axes = np.atleast_1d(axes).ravel()
present = set()
for ax, (arm, note, adata) in zip(axes, loaded):
    labels = adata.obs["pred_CT"].astype(str).to_numpy()
    xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
    present |= set(labels.tolist())
    cohort._scatter_on_he(
        ax, thumb, full_w, full_h, xy, labels, palette,
        f"{arm}\n{note}  n={adata.n_obs:,}",
        point_s,
        show_he=True,
    )
for ax in axes[len(loaded):]:
    ax.set_axis_off()
order = [name for name in palette if name in present]
order += sorted(present - set(order))
handles = [
    Line2D(
        [0], [0], linestyle="none", marker="o", markersize=8,
        markerfacecolor=palette.get(name, (0.5, 0.5, 0.5, 1))[:3],
        markeredgecolor="none", label=name,
    )
    for name in order
]
axes[len(loaded)].legend(handles=handles, loc="center", frameon=False, fontsize=8, ncol=1)
fig.suptitle("10x Block A Section 1  —  L2 cell type", fontsize=13)
fig.tight_layout()
out = BRCA / "pred_uni2_axisfix_figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / f"{SAMPLE}_L2_axisfix.png", dpi=160, bbox_inches="tight", facecolor="white")
plt.show()